# Demo 09 : Running Agents

Everything in module 09, run end to end : bounding the loop, the evaluator
families, guardrails on the input and the output, human in the loop, and
where the tokens go.

Runs against a local model by default. Only the last cell needs an account.

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

# Optional keys. A few notebooks call a third-party API : Tavily in demos05a,
# OpenWeatherMap in demos05b, LangSmith in demos09. load_dotenv() covers the
# local path ; in Colab there is no .env, so they are read from Secrets here.
# Missing is fine, the cell that needs one says so.
if IN_COLAB:
    try:
        from google.colab import userdata
        for _name in ("TAVILY_API_KEY", "OWM_API_KEY", "LANGSMITH_API_KEY"):
            try:
                _v = userdata.get(_name)
                if _v:
                    os.environ[_name] = _v
            except Exception:
                pass
    except ImportError:
        pass


In [ ]:
%pip install -q langsmith

## 1. Bounding the loop

A graph with an edge back to itself never reaches an end state. LangGraph
stops it rather than running forever.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.errors import GraphRecursionError


class Counter(TypedDict):
    n: int


def step(state: Counter) -> Counter:
    return {"n": state["n"] + 1}


builder = StateGraph(Counter)
builder.add_node("step", step)
builder.add_edge(START, "step")
builder.add_edge("step", "step")          # no way out
graph = builder.compile()

try:
    graph.invoke({"n": 0}, {"recursion_limit": 10})
except GraphRecursionError as exc:
    print(str(exc).split("For troubleshooting")[0].strip())

Recursion limit of 10 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.

The same bound expressed in terms of the agent instead of the graph. Both
limits here are per run; `thread_limit` bounds a whole conversation instead.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
)


def get_sla(customer: str) -> str:
    """Get the agreed response time for a customer."""
    return f"{customer} is on Enterprise: response within 1 hour."


bounded = create_agent(
    model=llm,
    tools=[get_sla],
    middleware=[
        ModelCallLimitMiddleware(run_limit=8, exit_behavior="end"),
        ToolCallLimitMiddleware(run_limit=6, exit_behavior="end"),
    ],
)

out = bounded.invoke({"messages": [{"role": "user", "content": "What is the SLA for Nova Logistics?"}]})
print(out["messages"][-1].content)

## 2. Evaluation : embedding distance

A string evaluator scores the wording. An embedding evaluator scores the
meaning, so a paraphrase costs almost nothing.

The score is a distance, so lower means closer.

In [ ]:
from langchain_classic.evaluation import load_evaluator

evaluator = load_evaluator("embedding_distance", embeddings=make_embeddings())

prediction = "Your refund has been approved and will arrive in 3 days."
references = [
    "Your refund has been approved and will arrive in 3 days.",
    "We have approved the refund; expect it within three working days.",
    "Your order has shipped and arrives on Thursday.",
]

for reference in references:
    score = evaluator.evaluate_strings(prediction=prediction, reference=reference)["score"]
    print(f"{score:.4f}   {reference}")

## 3. Evaluation : criteria

`labeled_criteria` grades an answer against a reference and a named rubric.
This is a model grading a model, so read the reasoning and not the score alone.

In [6]:
criteria_eval = load_evaluator("labeled_criteria", criteria="correctness", llm=llm)

for prediction in ["2 + 2 = 4", "2 + 2 = 5"]:
    result = criteria_eval.evaluate_strings(
        prediction=prediction, input="Calculate 2 + 2", reference="4"
    )
    print(prediction, "->", result["value"], result["score"])
    print("   ", " ".join(result["reasoning"].split())[:160])

2 + 2 = 4 -> Y 1
    Step 1: Analyze the input task. The task is to calculate the sum of 2 and 2. Step 2: Evaluate the correctness criterion. The submission states "2 + 2 = 4". The 


2 + 2 = 5 -> N 0
    Step 1: Analyze the input task. The task is to calculate the sum of 2 plus 2. Step 2: Evaluate the correctness criterion. The correct mathematical result of $2 


## 4. Evaluation : trajectory

The answer alone cannot tell a sound path from a lucky one. A trajectory
evaluator compares the steps taken against the steps expected.

In [7]:
trajectory_eval = load_evaluator("trajectory", llm=llm)

reference_steps = [
    "Fibonacci starts with 0, 1",
    "Each next number is the sum of the two previous ones",
    "The 5th number is 5",
]
prediction_steps = [
    "Fibonacci starts with 0, 1",
    "Each next number is the sum of the previous ones",
    "The 5th number is 4",
]

result = trajectory_eval.invoke({
    "question": "What is the 5th number of the Fibonacci sequence?",
    "answer": "4",
    "agent_trajectory": prediction_steps,
    "reference": {"answer": "5", "agent_trajectory": reference_steps},
})
print("score:", result["score"])
print(" ".join(result["reasoning"].split())[:400])

score: 0.5
### Step-by-Step Evaluation **i. Is the final answer helpful?** The final answer is "4". According to the provided ground truth, the correct answer is "5". Therefore, the answer is **incorrect**. While the number itself might seem plausible to a layperson (counting 0, 1, 1, 2, 3...), it fails the correctness check defined in the prompt. **ii. Does the AI language use a logical sequence of tools to


## 5. Input guardrail

`PIIMiddleware` rewrites personal data before the model ever sees it. Look at
the HumanMessage in the output: the address the user typed is gone.

In [8]:
from langchain.agents.middleware import PIIMiddleware


def lookup(order_id: str) -> str:
    """Look up an order by its id."""
    return f"Order {order_id} shipped."


guarded = create_agent(
    model=llm,
    tools=[lookup],
    middleware=[PIIMiddleware("email", strategy="redact")],
)

out = guarded.invoke({"messages": [{
    "role": "user",
    "content": "My email is jan@example.com, check order AB-9 for me.",
}]})

for message in out["messages"]:
    print(f"{type(message).__name__:14} | {(message.content or '')[:90]}")

HumanMessage   | My email is [REDACTED_EMAIL], check order AB-9 for me.
AIMessage      | 
ToolMessage    | Order AB-9 shipped.
AIMessage      | I've checked order AB-9, and it has been shipped. Is there anything else you'd like to kno


## 6. Output guardrail

Structural checks need no model and cost nothing. Run them before anything
downstream acts on the answer.

In [9]:
from langchain_classic.evaluation import (
    JsonValidityEvaluator,
    RegexMatchStringEvaluator,
)

json_eval = JsonValidityEvaluator()
print(json_eval.evaluate_strings(prediction='{x: 1}'))
print(json_eval.evaluate_strings(prediction='{"x": 1}'))

regex_eval = RegexMatchStringEvaluator()
pattern = r"^Order ID: [A-Z]{3}-\d{4}$"
print(regex_eval.evaluate_strings(prediction="Order ID: ABC-1234", reference=pattern))
print(regex_eval.evaluate_strings(prediction="somewhere in the warehouse", reference=pattern))

{'score': 0, 'reasoning': 'Expecting property name enclosed in double quotes: line 1 column 2 (char 1)'}
{'score': 1}
{'score': 1}
{'score': 0}


## 7. Human in the loop

The strongest guardrail on an irreversible action is a person. The agent stops
before the tool runs and hands back the pending call. A checkpointer is
required, because the run has to survive the pause.

In [10]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


def refund(order_id: str) -> str:
    """Refund an order."""
    return f"Refunded {order_id}."


approval_agent = create_agent(
    model=llm,
    tools=[refund],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"refund": True})],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "demo09"}}
out = approval_agent.invoke(
    {"messages": [{"role": "user", "content": "Refund order AB-9."}]}, config
)

pending = out["__interrupt__"][0].value["action_requests"][0]
print("waiting on:", pending["name"], pending["args"])

# a person decides here; approve, edit, reject or respond
out = approval_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
print("after approval:", out["messages"][-1].content)

waiting on: refund {'order_id': 'AB-9'}
after approval: Order AB-9 has been successfully refunded.


## 8. Where the tokens go

Cost in an agent is driven by the loop. Every pass resends
the whole history, and the tool schemas travel on every call.

In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback

with get_usage_metadata_callback() as cb:
    bounded.invoke({"messages": [{"role": "user", "content": "What is the SLA for Nova Logistics?"}]})
    print(cb.usage_metadata)

## 9. Tracing with LangSmith

Set the four variables before the agent is built and every run shows up in the
project at smith.langchain.com. Nothing else in the code changes.

This cell is skipped unless `LANGSMITH_API_KEY` is in our `.env`.

In [ ]:
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGSMITH_PROJECT"] = "running-agents"

    traced = create_agent(model=make_llm(), tools=[get_sla])
    result = traced.invoke({"messages": [{"role": "user", "content": "What is the SLA for Nova Logistics?"}]})
    print(result["messages"][-1].content)
    print("Open smith.langchain.com and look at the project 'running-agents'.")
else:
    print("No LANGSMITH_API_KEY in .env; skipping. Everything above ran without it.")